In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import seaborn as sns
import pandas as pd
import json
import os
from tensorflow import keras
import joblib

# Chargement des données de test
X_test = np.load("../data/processed/X_test.npy")
y_test = np.load("../data/processed/y_test.npy").astype(int)

classes = {0:"Normal", 1:"Supraventriculaire", 2:"Ventriculaire", 3:"Fusion", 4:"Inconnu"}

In [2]:
with open("../outputs/results/mitbih_metrics.json") as f:
    mit = json.load(f)
with open("../outputs/results/ptbdb_metrics.json") as f:
    ptb = json.load(f)

mit_df = pd.DataFrame(mit)[["model","auc_roc","auc_pr"]].rename(
    columns={"auc_roc":"AUC-ROC MIT","auc_pr":"AUC-PR MIT"})
ptb_df = pd.DataFrame(ptb)[["model","auc_roc","auc_pr"]].rename(
    columns={"auc_roc":"AUC-ROC PTB","auc_pr":"AUC-PR PTB"})

df = mit_df.merge(ptb_df, on="model")
df.columns = ["Modèle","AUC-ROC MIT-BIH","AUC-PR MIT-BIH","AUC-ROC PTBDB","AUC-PR PTBDB"]

print("=" * 70)
print("        TABLEAU COMPARATIF — MIT-BIH vs PTBDB")
print("=" * 70)
print(df.to_string(index=False))

# Graphique
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
x = np.arange(len(df))
w = 0.35

for ax, (col1, col2, title) in zip(axes, [
    ("AUC-ROC MIT-BIH","AUC-ROC PTBDB","AUC-ROC : MIT-BIH vs PTBDB"),
    ("AUC-PR MIT-BIH","AUC-PR PTBDB","AUC-PR : MIT-BIH vs PTBDB")
]):
    ax.bar(x-w/2, df[col1], w, label="MIT-BIH", color="#1A56DB", alpha=0.85)
    ax.bar(x+w/2, df[col2], w, label="PTBDB",   color="#7C3AED", alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(df["Modèle"], rotation=10, fontsize=10)
    ax.set_ylim(0, 1.05)
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.legend()
    ax.grid(axis="y", alpha=0.3)
    for i, (v1, v2) in enumerate(zip(df[col1], df[col2])):
        ax.text(i-w/2, v1+0.02, f"{v1:.3f}", ha="center", fontsize=9)
        ax.text(i+w/2, v2+0.02, f"{v2:.3f}", ha="center", fontsize=9)

plt.suptitle("Comparaison des performances — MIT-BIH vs PTBDB", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("../outputs/figures/comparaison_mitbih_ptbdb.png", dpi=150, bbox_inches="tight")
plt.show()
print("[✓] Figure sauvegardée")

        TABLEAU COMPARATIF — MIT-BIH vs PTBDB
           Modèle  AUC-ROC MIT-BIH  AUC-PR MIT-BIH  AUC-ROC PTBDB  AUC-PR PTBDB
Autoencoder Dense           0.8804          0.6366         0.5903        0.8165
 LSTM Autoencoder           0.9504          0.8270         0.6872        0.8605
 Isolation Forest           0.7532          0.3381         0.5402        0.7794
    One-Class SVM           0.7049          0.3839         0.6110        0.7994
[✓] Figure sauvegardée


C:\Users\HP\AppData\Local\Temp\ipykernel_16868\3448846140.py:43: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [3]:
# Chargement du LSTM Autoencoder
lstm_ae = keras.models.load_model("../outputs/models/lstm_autoencoder.keras")

def lstm_error(model, X):
    X_3d = X.reshape(X.shape[0], X.shape[1], 1)
    X_pred = model.predict(X_3d, verbose=0)
    return np.mean(np.power(X_3d - X_pred, 2), axis=(1,2))

errors = lstm_error(lstm_ae, X_test)
threshold = np.percentile(errors, 95)
y_pred = (errors > threshold).astype(int)
y_binary = (y_test != 0).astype(int)  # 0=normal, 1=anomalie

print(f"Seuil : {threshold:.4f}")
print(f"Anomalies réelles : {y_binary.sum()}")
print(f"Anomalies détectées : {y_pred.sum()}")

Seuil : 0.0200
Anomalies réelles : 3774
Anomalies détectées : 1095


In [4]:
print("\n" + "="*55)
print("  TAUX DE DÉTECTION PAR CLASSE — LSTM Autoencoder")
print("="*55)

resultats = []
for cls in range(5):
    idx = np.where(y_test == cls)[0]
    if len(idx) == 0:
        continue
    detected = y_pred[idx].sum()
    total = len(idx)
    rate = detected / total * 100
    label = "Normal" if cls == 0 else "Anomalie"
    resultats.append({
        "Classe": f"{cls} — {classes[cls]}",
        "Type": label,
        "Total": total,
        "Détectés anomalie": detected,
        "Taux détection %": round(rate, 1)
    })
    print(f"  Classe {cls} ({classes[cls]:20s}) : {detected:5d}/{total:5d} = {rate:5.1f}%")

# Graphique
df_cls = pd.DataFrame(resultats)
colors = ["#1A56DB" if "Normal" in t else "#DC2626" for t in df_cls["Type"]]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(df_cls["Classe"], df_cls["Taux détection %"], color=colors, alpha=0.85, edgecolor="white")
ax.axvline(x=50, color="gray", linestyle="--", alpha=0.5, label="50%")
ax.set_xlabel("Taux de détection comme anomalie (%)", fontsize=11)
ax.set_title("LSTM Autoencoder — Taux de détection par classe ECG", fontsize=12, fontweight="bold")
ax.set_xlim(0, 105)
for bar, val in zip(bars, df_cls["Taux détection %"]):
    ax.text(val+1, bar.get_y()+bar.get_height()/2, f"{val}%", va="center", fontsize=10)

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color="#1A56DB", label="Classe normale"),
    Patch(color="#DC2626", label="Classe anormale"),
], loc="lower right")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig("../outputs/figures/detection_par_classe.png", dpi=150, bbox_inches="tight")
plt.show()
print("[✓] Figure sauvegardée")


  TAUX DE DÉTECTION PAR CLASSE — LSTM Autoencoder
  Classe 0 (Normal              ) :    63/18118 =   0.3%
  Classe 1 (Supraventriculaire  ) :     8/  556 =   1.4%
  Classe 2 (Ventriculaire       ) :   379/ 1448 =  26.2%
  Classe 3 (Fusion              ) :     1/  162 =   0.6%
  Classe 4 (Inconnu             ) :   644/ 1608 =  40.0%
[✓] Figure sauvegardée


C:\Users\HP\AppData\Local\Temp\ipykernel_16868\2574025044.py:44: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
fig, axes = plt.subplots(2, 3, figsize=(15, 7))

# Vrais Positifs — anomalies bien détectées
vp_idx = np.where((y_binary==1) & (y_pred==1))[0]
# Faux Négatifs — anomalies manquées
fn_idx = np.where((y_binary==1) & (y_pred==0))[0]
# Faux Positifs — normaux mal classés
fp_idx = np.where((y_binary==0) & (y_pred==1))[0]

for i, idx in enumerate(vp_idx[:3]):
    axes[0,i].plot(X_test[idx], color="#1E8449", linewidth=1.5)
    axes[0,i].set_title(f"Vrai Positif\nClasse réelle: {classes[y_test[idx]]}\nErreur: {errors[idx]:.4f}", fontsize=9)
    axes[0,i].axhline(y=errors[idx], color="red", linestyle="--", alpha=0.3)
    axes[0,i].grid(True, alpha=0.3)

for i, idx in enumerate(fn_idx[:3]):
    axes[1,i].plot(X_test[idx], color="#C0392B", linewidth=1.5)
    axes[1,i].set_title(f"Faux Négatif (manqué)\nClasse réelle: {classes[y_test[idx]]}\nErreur: {errors[idx]:.4f}", fontsize=9)
    axes[1,i].grid(True, alpha=0.3)

plt.suptitle("Exemples de signaux ECG — Vrais Positifs vs Faux Négatifs\n(LSTM Autoencoder)", 
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("../outputs/figures/exemples_FP_FN.png", dpi=150, bbox_inches="tight")
plt.show()
print("[✓] Figure sauvegardée")

[✓] Figure sauvegardée


C:\Users\HP\AppData\Local\Temp\ipykernel_16868\3800774614.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
justifications = {
    "Autoencoder Dense (MLP)": [
        "Baseline simple et rapide à entraîner",
        "Efficace pour capturer des patterns globaux du signal",
        "Limitation : traite le signal comme un vecteur plat, ignore l'ordre temporel"
    ],
    "LSTM Autoencoder": [
        "Conçu pour les séquences temporelles — naturellement adapté à l'ECG",
        "Les cellules LSTM mémorisent les dépendances à long terme (ondes P→QRS→T)",
        "Meilleur modèle : AUC-ROC=0.95 grâce à la sensibilité temporelle"
    ],
    "Isolation Forest": [
        "Méthode ensembliste robuste, pas d'hypothèse sur la distribution des données",
        "Très efficace en haute dimension, entraînement rapide",
        "Limitation : score global, ne capture pas la structure temporelle"
    ],
    "One-Class SVM": [
        "Apprend une frontière explicite autour des données normales",
        "Kernel RBF permet une séparation non-linéaire dans l'espace des features",
        "Limitation : ne passe pas à l'échelle (sous-échantillonnage à 10k points requis)"
    ]
}

print("=" * 60)
print("   JUSTIFICATION DU CHOIX DES ARCHITECTURES")
print("=" * 60)
for model, points in justifications.items():
    print(f"\n {model}")
    for pt in points:
        print(f"   {pt}")

   JUSTIFICATION DU CHOIX DES ARCHITECTURES

 Autoencoder Dense (MLP)
   Baseline simple et rapide à entraîner
   Efficace pour capturer des patterns globaux du signal
   Limitation : traite le signal comme un vecteur plat, ignore l'ordre temporel

 LSTM Autoencoder
   Conçu pour les séquences temporelles — naturellement adapté à l'ECG
   Les cellules LSTM mémorisent les dépendances à long terme (ondes P→QRS→T)
   Meilleur modèle : AUC-ROC=0.95 grâce à la sensibilité temporelle

 Isolation Forest
   Méthode ensembliste robuste, pas d'hypothèse sur la distribution des données
   Très efficace en haute dimension, entraînement rapide
   Limitation : score global, ne capture pas la structure temporelle

 One-Class SVM
   Apprend une frontière explicite autour des données normales
   Kernel RBF permet une séparation non-linéaire dans l'espace des features
   Limitation : ne passe pas à l'échelle (sous-échantillonnage à 10k points requis)
